# Paper v3: all experiments (A100)

**Run:** Runtime → Change runtime type → **A100 GPU** (High-RAM if offered) → Runtime → **Run all** → allow Google Drive.
Everything is saved to `MyDrive/paper_v3_results` after every run. If Colab disconnects, press **Run all** again; finished runs are skipped.
Expected time on an A100 40 GB: about **3–5 hours**. The most important parts run first.

| Part | Models | Tested on |
|---|---|---|
| 1 | TF-IDF SVM, DistilBERT, BERT, RoBERTa (full fine-tuning) trained on all PROMISE, with original labels and with Dalpiaz relabelled labels | External industrial requirements (Dronology, ReqView, Leeds, WASP) **and** UStAI user stories |
| 2 | Qwen2.5 1.5B / 7B / 14B Instruct (32B on 80 GB GPUs), zero-shot and 8-shot, no training | PROMISE Task A + B, external, user stories |
| 3 | RoBERTa-large, DeBERTa-v3-large (full fine-tuning) | PROMISE (random + project-held-out, same folds as before), external, user stories |
| 4 | Qwen2.5-7B + LoRA classifier | PROMISE project-held-out, external, user stories |
| 5 | Tables: macro-F1, paired bootstrap CIs vs SVM, calibration, confidence-gate transfer, reproducibility check against your v2 run | — |

**User stories:** UStAI (Yamani, Baslyman & Ahmed, PROMISE 2025, doi:10.1145/3727582.3728689): 1,260 user stories for AI systems, manually annotated with the non-functional requirements they imply. Gold label = "the story implies at least one NFR". The stories were generated by LLMs and include AI-specific qualities (fairness, interpretability), which the paper must state. Your own 242-story annotation is **not** used (agreement was at chance level).

Your earlier runs (`bert_paper_results`, `paper_v2_results`) are found automatically in Drive, or downloaded from your GitHub repo if the repo is public.

**At the end** the notebook downloads `paper_v3_results.zip`. Push it to GitHub or send it. Don't edit or delete files in it. If something fails, send the error and the last lines printed above it.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers", "accelerate", "peft", "sentencepiece",
                "tiktoken", "protobuf", "scikit-learn", "scipy", "pandas", "matplotlib"], check=False)

import os, re, csv, json, time, glob, random, tarfile, zipfile, platform, shutil, urllib.request, warnings, gc
import numpy as np, pandas as pd, torch
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from scipy.optimize import minimize_scalar
import transformers
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, AutoModelForCausalLM,
                          get_linear_schedule_with_warmup)
warnings.filterwarnings("ignore"); transformers.logging.set_verbosity_error()
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 80); pd.set_option("display.max_rows", 300)

dev = "cuda" if torch.cuda.is_available() else "cpu"
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9 if dev == "cuda" else 0
BF16 = dev == "cuda" and torch.cuda.is_bf16_supported()
if dev == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True; torch.backends.cudnn.allow_tf32 = True
print("Device:", dev, "|", torch.cuda.get_device_name(0) if dev == "cuda" else "no GPU", f"| {GPU_GB:.0f} GB | bf16 {BF16}")

In [ ]:
# ---------------- Configuration ----------------
QUICK_TEST = os.environ.get("V3_QUICK_TEST", "0") == "1"      # leave False
HEAVY = GPU_GB >= 35

SEEDS = [42, 43, 44]
EPOCHS, BATCH, MAX_LEN = 5, 16, 64
BASE_MODELS = {"distilbert": ("distilbert-base-uncased", 5e-5), "bert": ("bert-base-uncased", 3e-5),
               "roberta": ("roberta-base", 3e-5)}
LARGE_MODELS = {"roberta-large": ("FacebookAI/roberta-large", 1e-5), "deberta-v3-large": ("microsoft/deberta-v3-large", 1e-5)}
PROMPT_LLMS = {"qwen1.5b": "Qwen/Qwen2.5-1.5B-Instruct", "qwen7b": "Qwen/Qwen2.5-7B-Instruct"}
if HEAVY: PROMPT_LLMS["qwen14b"] = "Qwen/Qwen2.5-14B-Instruct"
if GPU_GB >= 75: PROMPT_LLMS["qwen32b"] = "Qwen/Qwen2.5-32B-Instruct"
LORA = ("qwen7b-lora", "Qwen/Qwen2.5-7B"); LORA_LR, LORA_EPOCHS, LORA_R = 1e-4, 3, 16
K_SHOT, N_BOOT, LLM_BATCH = 8, 2000, 8
RUN = dict(part1=True, part2=dev == "cuda", part3=HEAVY, part4=HEAVY)

if QUICK_TEST:   # tiny stand-ins on a data subset: code-path check only
    tiny = "google/bert_uncased_L-2_H-128_A-2"
    BASE_MODELS = {"distilbert": (tiny, 5e-5)}; LARGE_MODELS = {"roberta-large": (tiny, 1e-4)}
    PROMPT_LLMS = {"qwen1.5b": "HuggingFaceTB/SmolLM2-135M-Instruct"}; LORA = ("qwen7b-lora", "HuggingFaceTB/SmolLM2-135M")
    SEEDS, EPOCHS, LORA_EPOCHS, N_BOOT = [42], 1, 1, 100
    K_SHOT, LLM_BATCH, BATCH = 2, 2, 4
    RUN = dict(part1=True, part2=True, part3=True, part4=True)

try:
    from google.colab import drive
    drive.mount("/content/drive"); BASE = "/content/drive/MyDrive"
except Exception:
    BASE = "."
OUT = f"{BASE}/paper_v3_results" + ("_QUICKTEST" if QUICK_TEST else "")
os.makedirs(OUT, exist_ok=True)

# Earlier runs: Drive first, then the public GitHub repo
def find_dir(cands, probe):
    return next((c for c in cands if os.path.exists(os.path.join(c, probe))), None)
BPR = find_dir([f"{BASE}/bert_paper_results", "bert_paper_results"], "A_project_distilbert_42.npz")
V2 = find_dir([f"{BASE}/paper_v2_results", f"{BASE}/paper_v2_result"], "ext_orig_svm_42.npz")
if (BPR is None or V2 is None) and not QUICK_TEST:
    try:
        if not os.path.exists("tearmpaper"):
            subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/RezZup/tearmpaper.git"], check=True, timeout=300)
        BPR = BPR or find_dir(["tearmpaper/bert_paper_results"], "A_project_distilbert_42.npz")
        V2 = V2 or find_dir(["tearmpaper/paper_v2_result"], "ext_orig_svm_42.npz")
    except Exception as e:
        print("GitHub fallback failed:", e)
if QUICK_TEST: BPR = V2 = None
print("Saving to:", OUT, "| earlier base runs:", BPR, "| v2 run:", V2)
print("Parts:", RUN, "| prompt LLMs:", list(PROMPT_LLMS))
json.dump(dict(python=platform.python_version(), torch=torch.__version__, transformers=transformers.__version__,
               gpu=torch.cuda.get_device_name(0) if dev == "cuda" else "cpu", gpu_gb=round(GPU_GB, 1), quick_test=QUICK_TEST,
               seeds=SEEDS, epochs=EPOCHS, batch=BATCH, max_len=MAX_LEN, base=BASE_MODELS, large=LARGE_MODELS,
               prompt_llms=PROMPT_LLMS, k_shot=K_SHOT, lora=dict(model=LORA, lr=LORA_LR, epochs=LORA_EPOCHS, r=LORA_R),
               parts=RUN, earlier_base_runs=BPR, v2_run=V2), open(f"{OUT}/environment.json", "w"), indent=1)

In [ ]:
# ---------------- Data ----------------
os.makedirs("data", exist_ok=True)
def fetch(url, path):
    if not os.path.exists(path): urllib.request.urlretrieve(url, path)
    return path

# PROMISE NFR (Zenodo 268542)
arff = "data/nfr/nfr.arff"
if not os.path.exists(arff):
    tarfile.open(fetch("https://zenodo.org/api/records/268542/files/nfr.tar/content", "data/nfr.tar")).extractall("data")
rows, on = [], False
for line in open(arff, encoding="utf-8", errors="ignore"):
    line = line.strip()
    if line.lower().startswith("@data"): on = True; continue
    if on and line and not line.startswith("%"): rows.append(next(csv.reader([line], quotechar="'", escapechar="\\")))
P = pd.DataFrame(rows, columns=["project", "text", "cls"]); P["project"] = P.project.astype(int); P["cls"] = P.cls.str.strip()
assert len(P) == 625

# Dalpiaz et al. RE'19 (Zenodo 3309669): relabelled PROMISE + external industrial sets
zipfile.ZipFile(fetch("https://zenodo.org/api/records/3309669/files/explainable-re/RE-2019-Materials-v0.zip/content",
                      "data/re19.zip")).extractall("data/re19")
TAG = glob.glob("data/re19/*/Manually tagged datasets")[0]
R = pd.read_csv(f"{TAG}/promise-reclass.csv", encoding="latin1")
assert (R.ProjectID.values == P.project.values).all() and (R.Class.values == P.cls.values).all()
P["fr"] = np.where(P.cls == "F", "FR", "NFR"); P["y_orig"] = (P.cls != "F").astype(int)
P["isF"], P["isQ"] = R.IsFunctional.values, R.IsQuality.values
E = pd.concat([pd.read_csv(f"{TAG}/{n}.csv", encoding="latin1").assign(dataset=n) for n in ["dronology", "reqview", "leeds", "wasp"]])
E = pd.DataFrame(dict(dataset=E.dataset.values, text=E.RequirementText.astype(str).str.strip().values,
                      isF=E.IsFunctional.astype(int).values, isQ=E.IsQuality.astype(int).values))
E = E[(E.isF + E.isQ) > 0].reset_index(drop=True)
E["kind"] = np.select([(E.isF == 1) & (E.isQ == 0), (E.isF == 0) & (E.isQ == 1)], ["onlyF", "onlyQ"], "FQ")

# UStAI user stories (github.com/asmayamani/UStAI)
up = fetch("https://raw.githubusercontent.com/asmayamani/UStAI/main/UStAI-annotated_updated.csv", "data/ustai.csv")
try: Uraw = pd.read_csv(up, encoding="utf-8")
except UnicodeDecodeError: Uraw = pd.read_csv(up, encoding="latin1")
U = pd.DataFrame(dict(text=Uraw["User story"].astype(str).str.strip(), system=Uraw["Abstract"].astype(str),
                      generator=Uraw["LLM"].astype(str), nfr=Uraw["Non-Functional Requirements"].fillna("").astype(str).str.strip()))
U = U.drop_duplicates("text").reset_index(drop=True)
U["system"] = U.system.factorize()[0]
U["isQ"] = (U.nfr != "").astype(int)
cats = U.nfr.str.lower().str.split(r"\s*[-,/]\s*")
PROMISE_LIKE = {"performance", "efficiency", "usability", "accessibility", "security", "privacy", "integrity", "scalability",
                "reliability", "availability", "maintainability", "portability", "safety", "interoperability"}
U["nfr_group"] = ["none" if q == 0 else ("promise_like" if set(c) & PROMISE_LIKE else "ai_specific_only") for q, c in zip(U.isQ, cats)]

if QUICK_TEST:
    P = P.sample(200, random_state=0).sort_index().reset_index(drop=True)
    E = E.groupby("dataset").head(20).reset_index(drop=True); U = U.sample(120, random_state=0).reset_index(drop=True)

norm = lambda s: re.sub(r"[^a-z0-9]+", " ", str(s).lower()).strip()
pn = set(P.text.map(norm))
ov_E, ov_U = int(E.text.map(norm).isin(pn).sum()), int(U.text.map(norm).isin(pn).sum())
assert ov_E == 0 and ov_U == 0, (ov_E, ov_U)
TEST = pd.concat([E.text, U.text], ignore_index=True).values      # every trained model predicts both test sets in one pass
nE = len(E)

TASKS = {"A": (P.copy(), "fr"), "B": (P[P.cls.isin(["US", "SE", "O", "PE"])].reset_index(drop=True), "cls")}
def get_folds(d, label, protocol, seed):
    y = d[label].values
    if protocol == "random": return list(StratifiedKFold(10, shuffle=True, random_state=seed).split(d, y))
    return list(StratifiedGroupKFold(5, shuffle=True, random_state=seed).split(d, y, groups=d.project.values))

tk = AutoTokenizer.from_pretrained("bert-base-uncased" if not QUICK_TEST else BASE_MODELS["distilbert"][0])
trunc = lambda t: float(np.mean([len(tk(x)["input_ids"]) > MAX_LEN for x in t]))
card = dict(promise=len(P), external=len(E), external_by_kind=E.groupby(["dataset", "kind"]).size().unstack(fill_value=0).to_dict(orient="index"),
            ustai=len(U), ustai_dropped_duplicates=int(len(Uraw) - len(U)) if not QUICK_TEST else None,
            ustai_with_nfr=int(U.isQ.sum()), ustai_groups=U.nfr_group.value_counts().to_dict(), ustai_systems=int(U.system.nunique()),
            ustai_by_generator=U.groupby("generator").isQ.agg(["size", "sum"]).to_dict(orient="index"),
            share_truncated_at_max_len=dict(promise=trunc(P.text), external=trunc(E.text), ustai=trunc(U.text)),
            text_overlap_with_promise=dict(external=ov_E, ustai=ov_U))
json.dump(card, open(f"{OUT}/data_card.json", "w"), indent=1, default=int); print(json.dumps(card, indent=1, default=int))

In [ ]:
# ---------------- Model code ----------------
def set_seed(s): random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
def free():
    gc.collect()
    if dev == "cuda": torch.cuda.empty_cache()

def train_loop(model, tok, tr_text, tr_y, te_text, n_cls, lr, epochs, params):
    enc = lambda t: tok(list(t), truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt")
    Xtr, Xte = enc(tr_text), enc(te_text); ytr = torch.tensor(np.asarray(tr_y))
    counts = np.bincount(tr_y, minlength=n_cls).clip(min=1)
    lossf = torch.nn.CrossEntropyLoss(weight=torch.tensor(len(tr_y) / (n_cls * counts), dtype=torch.float, device=dev))
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.01)
    steps = epochs * int(np.ceil(len(tr_y) / BATCH)); sch = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
    model.train()
    for ep in range(epochs):
        perm = torch.randperm(len(tr_y))
        for i in range(0, len(perm), BATCH):
            idx = perm[i:i + BATCH]
            out = model(input_ids=Xtr["input_ids"][idx].to(dev), attention_mask=Xtr["attention_mask"][idx].to(dev))
            lossf(out.logits.float(), ytr[idx].to(dev)).backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0); opt.step(); sch.step(); opt.zero_grad()
    model.eval(); lg = []
    with torch.no_grad():
        for i in range(0, len(te_text), 64):
            lg.append(model(input_ids=Xte["input_ids"][i:i + 64].to(dev), attention_mask=Xte["attention_mask"][i:i + 64].to(dev)).logits.float().cpu())
    return torch.cat(lg).numpy()

def finetune_predict(name, lr, tr_text, tr_y, te_text, n_cls, seed):
    set_seed(seed); tok = AutoTokenizer.from_pretrained(name)
    # .float(): transformers 5 loads fp16 checkpoints (e.g. deberta-v3-large) in fp16 -> NaN with AdamW. Train in fp32.
    model = AutoModelForSequenceClassification.from_pretrained(name, num_labels=n_cls).float().to(dev)
    lg = train_loop(model, tok, tr_text, tr_y, te_text, n_cls, lr, EPOCHS, list(model.parameters())); del model; free()
    assert np.isfinite(lg).all(), f"non-finite logits from {name}"; return lg

def lora_predict(name, tr_text, tr_y, te_text, n_cls, seed):
    from peft import LoraConfig, get_peft_model
    set_seed(seed); tok = AutoTokenizer.from_pretrained(name)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    model = AutoModelForSequenceClassification.from_pretrained(name, num_labels=n_cls, torch_dtype=torch.bfloat16 if BF16 else torch.float32)
    model.config.pad_token_id = tok.pad_token_id
    model = get_peft_model(model, LoraConfig(task_type="SEQ_CLS", r=LORA_R, lora_alpha=2 * LORA_R, lora_dropout=0.05,
                                             target_modules=["q_proj", "k_proj", "v_proj", "o_proj"])).to(dev)
    params = [p for p in model.parameters() if p.requires_grad]
    lg = train_loop(model, tok, tr_text, tr_y, te_text, n_cls, LORA_LR, LORA_EPOCHS, params); del model; free()
    assert np.isfinite(lg).all(), f"non-finite logits from {name}"; return lg

def svm_scores(tr_text, tr_y, te_text, n_cls=None, seed=None):
    vec = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, stop_words="english")
    clf = LinearSVC(C=1, class_weight="balanced").fit(vec.fit_transform(tr_text), tr_y)
    s = clf.decision_function(vec.transform(te_text)); return (np.stack([-s, s], 1) if s.ndim == 1 else s).astype(np.float32)

FIT = {"svm": svm_scores, **{k: (lambda mn, lr: lambda a, b, c, n, s: finetune_predict(mn, lr, a, b, c, n, s))(*v)
                              for k, v in {**BASE_MODELS, **LARGE_MODELS}.items()},
       LORA[0]: lambda a, b, c, n, s: lora_predict(LORA[1], a, b, c, n, s)}

def run_cv(task, protocol, name, seed):
    f = f"{OUT}/cv_{task}_{protocol}_{name}_{seed}.npz"
    if os.path.exists(f): return
    d, label = TASKS[task]; classes = sorted(d[label].unique()); y = d[label].map({c: i for i, c in enumerate(classes)}).values
    fold = np.zeros(len(d), int); lg = np.zeros((len(d), len(classes)), np.float32); t0 = time.time()
    for k, (tr, te) in enumerate(get_folds(d, label, protocol, seed)):
        fold[te] = k; lg[te] = FIT[name](d.text.values[tr], y[tr], d.text.values[te], len(classes), seed)
    np.savez(f, pred=lg.argmax(1), logits=lg, y=y, fold=fold, project=d.project.values)
    print(f"cv  {task} {protocol:7s} {name:17s} seed {seed} | macro-F1 {f1_score(y, lg.argmax(1), average='macro'):.3f} | {time.time()-t0:.0f}s")

def run_out(scheme, name, seed):
    # Train on all PROMISE; predict external requirements + user stories.
    f = f"{OUT}/out_{scheme}_{name}_{seed}.npz"
    if os.path.exists(f): return
    if scheme == "orig": tr_text, tr_y = P.text.values, P.y_orig.values
    else: keep = ((P.isF + P.isQ) > 0).values; tr_text, tr_y = P.text.values[keep], P.isQ.values[keep]
    t0 = time.time(); lg = FIT[name](tr_text, tr_y, TEST, 2, seed)
    np.savez(f, logits_E=lg[:nE], pred_E=lg[:nE].argmax(1), logits_U=lg[nE:], pred_U=lg[nE:].argmax(1), n_train=len(tr_y))
    m = E.kind.values != "FQ"
    print(f"out {scheme:7s} {name:17s} seed {seed} | ext macro-F1 {f1_score(E.isQ.values[m], lg[:nE][m].argmax(1), average='macro', labels=[0, 1], zero_division=0):.3f}"
          f" | stories qflag-F1 {f1_score(U.isQ.values, lg[nE:].argmax(1), zero_division=0):.3f} | {time.time()-t0:.0f}s")

In [ ]:
# ---------------- Part 1: SVM + base models (train on all PROMISE -> external + user stories) ----------------
if RUN["part1"]:
    for seed in SEEDS:
        for task in TASKS:
            for protocol in ["random", "project"]: run_cv(task, protocol, "svm", seed)
    for scheme in ["orig", "reclass"]:
        for seed in SEEDS:
            for name in ["svm"] + list(BASE_MODELS): run_out(scheme, name, seed)

In [ ]:
# ---------------- Part 2: prompted LLMs (no training), zero-shot and 8-shot ----------------
INSTR_A = ("Classify the software requirement.\n"
           "A) Functional: it describes a behaviour or function that the system must perform.\n"
           "B) Non-functional: it describes a quality or constraint of the system, such as performance, security, usability, "
           "availability, look and feel, legal, maintainability, portability, scalability, operability or fault tolerance.")
INSTR_B = ("Classify the non-functional software requirement into one quality class.\n"
           "A) Operability: how the system is operated, installed, administered or supported in its environment.\n"
           "B) Performance: speed, response time, throughput, capacity or resource use.\n"
           "C) Security: access control, authentication, privacy, integrity or protection against misuse.\n"
           "D) Usability: ease of use, learnability, user satisfaction or accessibility for users.")
LET = {"A": ["A", "B"], "B": ["A", "B", "C", "D"]}
def make_prompt(instr, letters, text, shots):
    s = instr + "\n\n" + ("Examples:\n" + "".join(f'Requirement: "{t}"\nAnswer: {letters[c]}\n\n' for t, c in shots) if shots else "")
    ask = " or ".join(letters) if len(letters) == 2 else ", ".join(letters[:-1]) + " or " + letters[-1]
    return s + f'Requirement: "{text}"\nAnswer with the single letter {ask}.'
def pick_shots(texts, y, n_cls, k, rng):
    idx = []
    for c in range(n_cls):
        pool = np.where(y == c)[0]; idx += list(rng.choice(pool, min(k // n_cls, len(pool)), replace=False))
    rng.shuffle(idx); return [(texts[i], int(y[i])) for i in idx]

class Scorer:
    def __init__(self, name):
        self.tok = AutoTokenizer.from_pretrained(name); self.tok.padding_side = "left"
        if self.tok.pad_token is None: self.tok.pad_token = self.tok.eos_token
        self.model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.bfloat16 if BF16 else torch.float32).to(dev).eval()
    def __call__(self, prompts, letters, batch=None):
        batch = batch or LLM_BATCH
        ids = [self.tok.encode(l, add_special_tokens=False)[0] for l in letters]; assert len(set(ids)) == len(ids)
        out = []
        for i in range(0, len(prompts), batch):
            msgs = [self.tok.apply_chat_template([{"role": "user", "content": p}], tokenize=False, add_generation_prompt=True) for p in prompts[i:i + batch]]
            x = self.tok(msgs, return_tensors="pt", padding=True, add_special_tokens=False).to(dev)
            pos = (x["attention_mask"].cumsum(-1) - 1).clamp(min=0)
            with torch.no_grad():
                try: lg = self.model(**x, position_ids=pos, logits_to_keep=1).logits[:, -1, :].float()
                except TypeError: lg = self.model(**x, position_ids=pos).logits[:, -1, :].float()
            out.append(lg[:, ids].cpu().numpy())
        return np.concatenate(out)

if RUN["part2"]:
    for short, name in PROMPT_LLMS.items():
        todo = [(m, s) for m, s in [("zs", 0)] + [(f"fs{K_SHOT}", s) for s in SEEDS] if not os.path.exists(f"{OUT}/llm_{short}-{m}_{s}.npz")]
        if not todo: continue
        t0 = time.time(); sc = Scorer(name); print("loaded", name, f"{time.time()-t0:.0f}s")
        for mode, seed in todo:
            t0 = time.time(); res = {}
            for task in ["A", "B"]:
                d, label = TASKS[task]; classes = sorted(d[label].unique()); y = d[label].map({c: i for i, c in enumerate(classes)}).values
                instr = INSTR_A if task == "A" else INSTR_B; fold = np.zeros(len(d), int); scores = np.zeros((len(d), len(classes)), np.float32)
                if mode == "zs":    # folds recorded only for cross-fitted calibration
                    for k, (_, te) in enumerate(get_folds(d, label, "project", SEEDS[0])): fold[te] = k
                    scores[:] = sc([make_prompt(instr, LET[task], t, None) for t in d.text.values], LET[task])
                else:               # 8 examples drawn only from the other projects (training folds)
                    rng = np.random.default_rng(seed)
                    for k, (tr, te) in enumerate(get_folds(d, label, "project", seed)):
                        fold[te] = k; shots = pick_shots(d.text.values[tr], y[tr], len(classes), K_SHOT, rng)
                        scores[te] = sc([make_prompt(instr, LET[task], t, shots) for t in d.text.values[te]], LET[task])
                res.update({f"{task}_scores": scores, f"{task}_y": y, f"{task}_fold": fold, f"{task}_project": d.project.values})
            shots = None if mode == "zs" else pick_shots(P.text.values, P.y_orig.values, 2, K_SHOT, np.random.default_rng(seed + 1000))
            s_out = sc([make_prompt(INSTR_A, LET["A"], t, shots) for t in TEST], LET["A"])
            np.savez(f"{OUT}/llm_{short}-{mode}_{seed}.npz", **res, logits_E=s_out[:nE], logits_U=s_out[nE:])
            m = E.kind.values != "FQ"
            print(f"llm {short}-{mode} seed {seed} | A {f1_score(res['A_y'], res['A_scores'].argmax(1), average='macro'):.3f} "
                  f"| B {f1_score(res['B_y'], res['B_scores'].argmax(1), average='macro'):.3f} "
                  f"| ext {f1_score(E.isQ.values[m], s_out[:nE][m].argmax(1), average='macro', labels=[0, 1], zero_division=0):.3f} "
                  f"| stories qflag-F1 {f1_score(U.isQ.values, s_out[nE:].argmax(1), zero_division=0):.3f} | {time.time()-t0:.0f}s")
        del sc; free()

In [ ]:
# ---------------- Part 3: large encoders ----------------
if RUN["part3"]:
    for seed in SEEDS:
        for name in LARGE_MODELS:
            for scheme in ["orig", "reclass"]: run_out(scheme, name, seed)
            for task in TASKS:
                for protocol in ["project", "random"]: run_cv(task, protocol, name, seed)

In [ ]:
# ---------------- Part 4: Qwen2.5-7B + LoRA classifier ----------------
if RUN["part4"]:
    for seed in SEEDS:
        for scheme in ["orig", "reclass"]: run_out(scheme, LORA[0], seed)
        for task in TASKS: run_cv(task, "project", LORA[0], seed)

In [ ]:
# ---------------- Part 5a: collect all runs ----------------
def softmax(z, T=1.0):
    z = np.asarray(z, float) / T; z = z - z.max(1, keepdims=True); e = np.exp(z); return e / e.sum(1, keepdims=True)
runs = []
def add(domain, protocol, model, seed, y, logits, group, fold=None, has_prob=True):
    logits = np.asarray(logits, float)
    runs.append(dict(domain=domain, protocol=protocol, model=model, seed=int(seed), y=np.asarray(y), pred=logits.argmax(1),
                     logits=logits if has_prob else None, group=np.asarray(group), fold=fold))
for f in sorted(glob.glob(f"{OUT}/cv_*.npz")):
    _, task, protocol, model, seed = os.path.basename(f)[:-4].split("_"); z = np.load(f)
    add(task, protocol, model, seed, z["y"], z["logits"], z["project"], z["fold"], model != "svm")
have = {(r["domain"], r["protocol"], r["model"], r["seed"]) for r in runs}
if BPR:   # earlier full fine-tuning runs of the base models (same folds)
    for f in sorted(glob.glob(f"{BPR}/[AB]_*_*_*.npz")):
        task, protocol, model, seed = os.path.basename(f)[:-4].split("_")
        if (task, protocol, model, int(seed)) in have or model == "svm": continue
        z = np.load(f); add(task, protocol, model, seed, z["y"], z["logits"], z["project"], z["fold"])
for f in sorted(glob.glob(f"{OUT}/out_*.npz")):
    _, scheme, model, seed = os.path.basename(f)[:-4].split("_"); z = np.load(f)
    add("ext", scheme, model, seed, E.isQ.values, z["logits_E"], E.dataset.values, None, model != "svm")
    add("us", scheme, model, seed, U.isQ.values, z["logits_U"], U.system.values, None, model != "svm")
for f in sorted(glob.glob(f"{OUT}/llm_*.npz")):
    model, seed = os.path.basename(f)[4:-4].rsplit("_", 1); z = np.load(f)
    for task in ["A", "B"]: add(task, "project", model, seed, z[f"{task}_y"], z[f"{task}_scores"], z[f"{task}_project"], z[f"{task}_fold"])
    add("ext", "orig", model, seed, E.isQ.values, z["logits_E"], E.dataset.values)
    add("us", "orig", model, seed, U.isQ.values, z["logits_U"], U.system.values)
print(len(runs), "runs collected")

In [ ]:
# ---------------- Part 5b: tables ----------------
def mf1(y, p, labels=None): return f1_score(y, p, average="macro", labels=labels, zero_division=0)
def fmt(g, col):
    t = g.agg(mean=(col, "mean"), sd=(col, "std"), n=("seed", "nunique")).reset_index()
    t["cell"] = [f"{m:.3f} ± {s:.3f}" if n > 1 else f"{m:.3f} (1 run)" for m, s, n in zip(t["mean"], t["sd"], t["n"])]; return t

IN = pd.DataFrame([dict(task=r["domain"], protocol=r["protocol"], model=r["model"], seed=r["seed"], macro_f1=mf1(r["y"], r["pred"]),
                        acc=(r["y"] == r["pred"]).mean()) for r in runs if r["domain"] in ("A", "B")])
IN.to_csv(f"{OUT}/indomain_per_seed.csv", index=False)
T1 = fmt(IN.groupby(["model", "task", "protocol"]), "macro_f1").pivot(index="model", columns=["task", "protocol"], values="cell")
T1.to_csv(f"{OUT}/T1_indomain_macro_f1.csv"); print("T1 PROMISE in-domain macro-F1 (LLM rows: no training; 8-shot examples from other projects)"); print(T1)

def ext_m(y, p, kind):
    m = kind != "FQ"
    return dict(macroF1=mf1(y[m], p[m], [0, 1]), qflag_P=precision_score(y, p, zero_division=0), qflag_R=recall_score(y, p, zero_division=0),
                qflag_F1=f1_score(y, p, zero_division=0), FQ_flagged=p[kind == "FQ"].mean(), onlyF_flagged=p[kind == "onlyF"].mean())
def us_m(y, p, grp):
    return dict(macroF1=mf1(y, p, [0, 1]), qflag_P=precision_score(y, p, zero_division=0), qflag_R=recall_score(y, p, zero_division=0),
                qflag_F1=f1_score(y, p, zero_division=0), R_promise_like=p[grp == "promise_like"].mean(),
                R_ai_specific_only=p[grp == "ai_specific_only"].mean(), false_flag_no_nfr=p[grp == "none"].mean())
rows_e, rows_u = [], []
for r in runs:
    if r["domain"] == "ext":
        rows_e.append(dict(scheme=r["protocol"], model=r["model"], seed=r["seed"], dataset="ALL", **ext_m(r["y"], r["pred"], E.kind.values)))
        for ds in E.dataset.unique():
            k = E.dataset.values == ds
            rows_e.append(dict(scheme=r["protocol"], model=r["model"], seed=r["seed"], dataset=ds, **ext_m(r["y"][k], r["pred"][k], E.kind.values[k])))
    if r["domain"] == "us":
        rows_u.append(dict(scheme=r["protocol"], model=r["model"], seed=r["seed"], generator="ALL", **us_m(r["y"], r["pred"], U.nfr_group.values)))
        for g in U.generator.unique():
            k = U.generator.values == g
            rows_u.append(dict(scheme=r["protocol"], model=r["model"], seed=r["seed"], generator=g, **us_m(r["y"][k], r["pred"][k], U.nfr_group.values[k])))
XE, XU = pd.DataFrame(rows_e), pd.DataFrame(rows_u)
XE.to_csv(f"{OUT}/ext_per_seed.csv", index=False); XU.to_csv(f"{OUT}/stories_per_seed.csv", index=False)
T2 = XE[XE.dataset == "ALL"].drop(columns=["dataset", "seed"]).groupby(["scheme", "model"]).agg(["mean", "std"]).round(3)
T2.to_csv(f"{OUT}/T2_external.csv"); print("\nT2 External requirements (macroF1 on onlyF vs onlyQ items; qflag = has a quality aspect, all items)"); print(T2)
T2b = XE[XE.dataset != "ALL"].groupby(["scheme", "model", "dataset"]).macroF1.mean().round(3).unstack("dataset")
T2b.to_csv(f"{OUT}/T2b_external_per_dataset.csv"); print(T2b)
T3 = XU[XU.generator == "ALL"].drop(columns=["generator", "seed"]).groupby(["scheme", "model"]).agg(["mean", "std"]).round(3)
T3.to_csv(f"{OUT}/T3_user_stories.csv"); print("\nT3 UStAI user stories (gold: story implies >= 1 NFR)"); print(T3)
T3b = XU[XU.generator != "ALL"].groupby(["scheme", "model", "generator"]).qflag_F1.mean().round(3).unstack("generator")
T3b.to_csv(f"{OUT}/T3b_user_stories_by_generator.csv"); print(T3b)

In [ ]:
# ---------------- Part 5c: paired bootstrap CIs vs SVM ----------------
# PROMISE: resample whole projects, per seed.  External: resample within each dataset.  User stories: resample whole
# AI systems (42 clusters).  External/stories use the seed-averaged statistic.  CI containing 0 = no evidence of a difference.
rng = np.random.default_rng(0)
svm = {(r["domain"], r["protocol"], r["seed"]): r for r in runs if r["model"] == "svm"}
rows = []
for r in runs:
    if r["domain"] not in ("A", "B") or r["model"] == "svm": continue
    b = svm.get((r["domain"], r["protocol"], r["seed"] if r["seed"] in SEEDS else SEEDS[0]))
    if b is None: continue
    assert (b["y"] == r["y"]).all()
    y, labels = r["y"], np.unique(r["y"]); by = {p: np.where(r["group"] == p)[0] for p in np.unique(r["group"])}; ks = list(by); d = []
    for _ in range(N_BOOT // 2):
        ii = np.concatenate([by[p] for p in rng.choice(ks, len(ks))]); d.append(mf1(y[ii], r["pred"][ii], labels) - mf1(y[ii], b["pred"][ii], labels))
    lo, hi = np.percentile(d, [2.5, 97.5])
    rows.append(dict(domain=r["domain"], protocol=r["protocol"], model=r["model"], seed=r["seed"], metric="macroF1",
                     diff_vs_svm=round(mf1(y, r["pred"], labels) - mf1(y, b["pred"], labels), 3), ci_low=round(lo, 3), ci_high=round(hi, 3), excludes_zero=bool(lo > 0 or hi < 0)))
def boot_group(domain, stat_fn, resampler):
    groups = {}
    for r in runs:
        if r["domain"] == domain: groups.setdefault((r["protocol"], r["model"]), []).append(r["pred"])
    idx = [resampler() for _ in range(N_BOOT)]
    for (scheme, model), A in groups.items():
        B = groups.get((scheme, "svm"))
        if model == "svm" or B is None: continue
        st = lambda ii: np.mean([stat_fn(ii, a) for a in A]) - np.mean([stat_fn(ii, b) for b in B])
        d = np.array([st(ii) for ii in idx]); lo, hi = np.percentile(d, [2.5, 97.5]); full = np.arange(len(A[0]))
        rows.append(dict(domain=domain, protocol=scheme, model=model, seed="mean", metric=stat_fn.__name__, diff_vs_svm=round(st(full), 3),
                         ci_low=round(lo, 3), ci_high=round(hi, 3), excludes_zero=bool(lo > 0 or hi < 0)))
byE = [np.where(E.dataset.values == d)[0] for d in E.dataset.unique()]
mE = E.kind.values != "FQ"
def ext_macroF1(ii, p): m = mE[ii]; return mf1(E.isQ.values[ii][m], p[ii][m], [0, 1])
boot_group("ext", ext_macroF1, lambda: np.concatenate([rng.choice(v, len(v)) for v in byE]))
byU = [np.where(U.system.values == s)[0] for s in U.system.unique()]
def us_qflag_F1(ii, p): return f1_score(U.isQ.values[ii], p[ii], zero_division=0)
def us_macroF1(ii, p): return mf1(U.isQ.values[ii], p[ii], [0, 1])
rs = lambda: np.concatenate([byU[i] for i in rng.choice(len(byU), len(byU))])
boot_group("us", us_qflag_F1, rs); boot_group("us", us_macroF1, rs)
T4 = pd.DataFrame(rows); T4.to_csv(f"{OUT}/T4_bootstrap_vs_svm.csv", index=False); print(T4.to_string(index=False))

In [ ]:
# ---------------- Part 5d: calibration and confidence-gate transfer ----------------
def ece(p, y, bins=15):
    conf, acc = p.max(1), p.argmax(1) == y; e = 0.0
    for lo, hi in zip(np.linspace(0, 1, bins + 1)[:-1], np.linspace(0, 1, bins + 1)[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return e
def fit_T(z, y): return minimize_scalar(lambda T: -np.log(softmax(z, T)[np.arange(len(y)), y] + 1e-12).mean(), bounds=(0.05, 20), method="bounded").x
GRID = np.round(np.arange(0.50, 0.996, 0.005), 3)
def pick_tau(conf, ok, target=0.95, minn=20):
    for t in GRID:
        m = conf >= t
        if m.sum() >= minn and ok[m].mean() >= target: return t
    return None
oof = {(r["model"], r["seed"]): r for r in runs if r["domain"] == "A" and r["protocol"] == "project" and r["logits"] is not None}
rows = []
for r in runs:
    if r["logits"] is None: continue
    if r["domain"] in ("A", "B"):
        y, z, f = r["y"], r["logits"], r["fold"]; pc = np.zeros_like(z)
        for k in np.unique(f): m = f == k; pc[m] = softmax(z[m], fit_T(z[~m], y[~m]))
        conf, ok = pc.max(1), pc.argmax(1) == y; G = np.zeros(len(y), bool)
        for k in np.unique(f):
            t = pick_tau(conf[f != k], ok[f != k]); G[f == k] = (conf[f == k] >= t) if t else False
        rows.append(dict(domain=r["domain"], protocol=r["protocol"], model=r["model"], seed=r["seed"], ECE_raw=ece(softmax(z), y), ECE_cal=ece(pc, y),
                         gate_cov=G.mean(), gate_prec=ok[G].mean() if G.any() else np.nan))
    elif r["protocol"] == "orig" and (r["model"], r["seed"]) in oof:     # T and tau from PROMISE Task A project OOF, applied unchanged
        s = oof[(r["model"], r["seed"])]; T = fit_T(s["logits"], s["y"]); po = softmax(s["logits"], T); tau = pick_tau(po.max(1), po.argmax(1) == s["y"])
        m = (E.kind.values != "FQ") if r["domain"] == "ext" else np.ones(len(r["y"]), bool)
        y, z = r["y"][m], r["logits"][m]; pc = softmax(z, T); conf, ok = pc.max(1), pc.argmax(1) == y
        g = (conf >= tau) if tau else np.zeros(len(y), bool)
        rows.append(dict(domain=r["domain"], protocol="orig", model=r["model"], seed=r["seed"], ECE_raw=ece(softmax(z), y), ECE_cal=ece(pc, y),
                         tau_from_PROMISE=tau, gate_cov=g.mean(), gate_prec=ok[g].mean() if g.any() else np.nan))
CAL = pd.DataFrame(rows); CAL.to_csv(f"{OUT}/calibration_per_seed.csv", index=False)
num = [c for c in ["ECE_raw", "ECE_cal", "tau_from_PROMISE", "gate_cov", "gate_prec"] if c in CAL]
CAL[num] = CAL[num].apply(pd.to_numeric, errors="coerce")
T5 = CAL.groupby(["domain", "protocol", "model"])[num].agg(["mean", "std"]).round(3)
T5.to_csv(f"{OUT}/T5_calibration_gate.csv"); print(T5)

In [ ]:
# ---------------- Part 5e: reproducibility check against your v2 run (same code, same seeds, different session) ----------------
if V2:
    rows = []
    for f in sorted(glob.glob(f"{V2}/ext_*_*_*.npz")):
        _, scheme, model, seed = os.path.basename(f)[:-4].split("_"); g = f"{OUT}/out_{scheme}_{model}_{seed}.npz"
        if not os.path.exists(g): continue
        a, b = np.load(f, allow_pickle=True)["pred"], np.load(g)["pred_E"]; m = E.kind.values != "FQ"
        rows.append(dict(scheme=scheme, model=model, seed=int(seed), same_predictions=float((a == b).mean()),
                         macroF1_v2=round(mf1(E.isQ.values[m], a[m], [0, 1]), 3), macroF1_v3=round(mf1(E.isQ.values[m], b[m], [0, 1]), 3)))
    REP = pd.DataFrame(rows); REP.to_csv(f"{OUT}/T6_reproducibility_v2_vs_v3.csv", index=False); print(REP.to_string(index=False))
else:
    print("v2 run not found; reproducibility check skipped")

In [ ]:
# ---------------- Package ----------------
zipname = "paper_v3_results" + ("_QUICKTEST" if QUICK_TEST else "")
shutil.make_archive(zipname, "zip", OUT); print("Created", zipname + ".zip |", len(os.listdir(OUT)), "files")
try:
    from google.colab import files; files.download(zipname + ".zip")
except Exception:
    print("Download it from the Colab file panel or Drive:", OUT)